# 04-зертханалық жұмыс. Топтық есептер, жазба саны мен орташа мәндердің жиынтық кестелері

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 3-дәріс пен зертханалық жұмыс; groupby, жиіліктер, үлестер және топ көлемін түсіну.

## Оқу мақсаттары

- Агрегатты аналитикалық сұраққа сай таңдау.
- Атаулы агрегаттар мен жиынтық кестелер құру.
- Нөлдік санауышты, деректердің жоқтығын және жалпы қорытындыны ажырату.

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `numpy`, `pandas`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.

Оқулықтың архивін ашып, `notebooks/` және `data/` қалталарын бірге сақтаңыз. Ноутбукты бөлек жүктесеңіз, Jupyter немесе Colab файлдар панелінде ағымдағы жұмыс қалтасында `data` ішкі қалтасын жасап, оқулықтан алынған `football.csv` файлын сонда орналастырыңыз. Бірінші ұяшық дерек жолын тексереді.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if not (project_dir / "data" / "football.csv").is_file():
    raise FileNotFoundError(
        "data/football.csv табылмады. Оқулықтан football.csv файлын жүктеп алып, "
        "ағымдағы жұмыс қалтасындағы data ішкі қалтасына орналастырыңыз."
    )

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Практикалық міндет

Футболшылар туралы тарихи деректер үшін топтық есептер мен жиынтық кестелер жинағын құрыңыз. Клубтар бойынша Wage орташа мәні мен медианасын салыстырып, таңдалған санат үйлесімдерін зерттеп, «клуб — позиция» санақ жиынтығын және орташа SprintSpeed жиынтығын құру қажет. Агрегаттардың дұрыс мағынасына және жоқ үйлесімдерді өңдеуге баса назар аударыңыз.

Жұмыс бастапқы 4-зертханалық жұмысты дамытады. Онда Chelsea, FC Barcelona, AS Monaco клубтары, Argentina, Russia шартты таңдамалары және 20/30 жасы сақталған. Кез келген жоқ үйлесім жолдардың болмауы ретінде көрсетілуі тиіс. Толық жауап алу үшін жоқ максимум не орташа мән орнына нөл қоюға болмайды.

## Дайындық және деректер

Python, pandas, NumPy және жинақтағы data/football.csv қажет. Wage пен Value дереккөз бірліктерімен берілген, валюта мен кезең расталмаған. SprintSpeed, Strength, Balance — физикалық өлшемдер емес, сипаттама рейтингтері. Penalties — голдар емес, пенальти орындау дағдысының рейтингі. Жиын тарихи; клубтардың қамтылуы туралы мәлімет файлға қатысты.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

football = pd.read_csv(Path('data') / 'football.csv')
result_dir = Path('work_results')
result_dir.mkdir(exist_ok=True)
assert football.shape == (12897, 42)



Әр есептің алдында оның деңгейін жазыңыз: бір жол — клуб, «клуб — позиция» жұбы немесе «ұлттық санат — жас» жұбы. Жиынтық кестеге өткенде қандай өрістер жол, баған және мән болғанын көрсетіңіз. Әйтпесе барлық ойыншының орташа рейтингі туралы сұрақты клубтық көрсеткіштердің орташа мәні туралы сұрақпен шатастыру оңай.

## Талданған мысал



In [ ]:
demo = pd.DataFrame({
    'Team': ['R', 'R', 'R', 'S'],
    'Role': ['GK', 'ST', 'ST', 'GK'],
    'Score': [30, 60, 90, 50]
})
counts = demo.pivot_table(
    index='Team', columns='Role', values='Score',
    aggfunc='size', fill_value=0, observed=True
)
means = demo.pivot_table(
    index='Team', columns='Role', values='Score',
    aggfunc='mean', observed=True
)
print(counts)
print(means)



Санақ кестесінде R үшін GK = 1, ST = 2; S үшін GK = 1, ST = 0. Орташа мәндер кестесінде R-ST 75-ке тең, ал S-ST анықталмаған. Санауыштағы нөл жазбалардың жоқтығын білдіреді. Орташа мәндегі нөл нақты өлшенген орташа 0 мәнін білдіріп, деректерді бұрмалар еді. Бұл айырманы өз жиынтық кестелеріңізде сақтау қажет.

## Тапсырмалар

### 1. Клубтар бойынша Wage орташа мәні мен медианасы

Атаулы агрегаттау арқылы Club, n_players, wage_mean, wage_median, wage_min, wage_max өрістері бар `club_metrics` құрыңыз. rtol=1e-10 және atol=1e-8 бар `np.isclose` ережесі бойынша mean мен median жақын клубтарды табыңыз; сәйкестіктер санын бөлек беріңіз. Нәтижені дәл теңдікпен салыстырып, айырманың барын не жоғын түсіндіріңіз. mean мен median сәйкестігін барлық ойыншының Wage мәні бірдей деп атамаңыз.

### 2. Сәйкес топтар арасындағы максимум

1-тапсырмадағы орташа мәні мен медианасы сәйкес клубтар арасынан максимал wage_mean және осы мәні бар барлық клубты табыңыз. Қорытындыны тексеру үшін n_players, wage_min және wage_max көрсетіңіз. Іріктеуден кейін жол қалмаса, оны тіркеп, бос кестенің бірінші жолына жүгінбеңіз. Бір жазбалық топтардың mean және median мәндері неліктен автоматты тең болатынын бөлек түсіндіріңіз.

### 3. Chelsea Wage қосындысы

groupby арқылы әр Club үшін Wage қосындысын алып, Chelsea көрсеткішін шығарыңыз. Оны бастапқы кестені тікелей сүзгілеу арқылы қайта есептеңіз. Қосындымен бірге Chelsea жолдарының санын және орташа Wage көрсетіңіз. Жоғары Wage қосындысы топтың үлкен көлемінен неліктен туындауы мүмкін екенін тұжырымдаңыз. Метадеректерде жоқ валюта мен кезеңді көрсетпеңіз.

### 4. Argentina: ұлттық санат және жас

Nationality және Age бойынша жол саны, Wage минимумы мен максимумы бар жазық есеп құрыңыз. Одан Argentina үшін Age == 20 кезіндегі максимал Wage, Age == 30 кезіндегі максимал және минимал Wage алыңыз. Әр жағдай үшін топтың бар-жоғын көрсетіңіз. Бір ұяшықты тікелей сүзгілеумен тексеріңіз. Деректердің болашақ нұсқасында үйлесім болмаса, дұрыс алгоритм апатты аяқталмай, түсіндірме қайтаруы тиіс.

### 5. Таңдалған құрамның екі сипаттамасы

Club == 'FC Barcelona' және Nationality == 'Argentina' үшін Strength максимумы мен Balance максимумын анықтаңыз. Әр максимумға жеткен барлық жолды Name және екі рейтингпен шығарыңыз. Максимумдардың бір бастапқы жазбаға тиесілігін тексеріңіз. Максимал Strength пен максимал Balance бір спортшыға тиесілі болуы міндетті емес; максимумдар кестесі өздігінен екі сипаттамасы да бар нақты «мінсіз ойыншыны» анықтамайды.

### 6. Клубтар мен позициялар бойынша санақ жиынтығы

`club_position_counts` құрыңыз: index = Club, columns = Position, мәндер — бастапқы жолдар саны. Бос үйлесімдерді нөлмен толтырыңыз. Барлық санауыш қосындысы len(football) мәніне, ал әр жол қосындысы тиісті клуб жазбаларының санына тең екенін тексеріңіз. Нөлдік мәндерді қоса алғанда, қамтылған барлық клуб бойынша GK орташа санын табыңыз; шығарылымды үш таңбаға дейін дөңгелектеңіз. CM санауышы нөл клубтар санын анықтаңыз. Бұл нақты клубта жартылай қорғаушылар жоқ екенінің дәлелі емес, файлда CM жазбаларының жоқтығы.

### 7. Wage бойынша Nationality — Club жиынтығы

Nationality және Club бойынша Wage қосындыларының жиынтық кестесін құрыңыз. Үйлесім бар болса, AS Monaco үшін Russia көрсеткішін `.loc` арқылы алыңыз. Тікелей іріктеумен салыстырып, қатысқан жол санын көрсетіңіз. Таңдалған міндетте бос қосындыны нөлмен толтыруға бола ма және бұл орташа мәнді нөлмен толтырудан қалай ерекшеленеді, түсіндіріңіз. Есепте бақылаулардың жоқтығы фактісін және таңдалған сандық көрсетілімді қатар сақтаңыз.

### 8. Орташа SprintSpeed жиынтық кестесі

Club пен Position қиылысындағы орташа SprintSpeed бар `club_position_speed` құрыңыз. Бос үйлесімдерді жетіспейтін мән ретінде сақтаңыз. Position бойынша тікелей groupby қолданып, барлық бастапқы ойыншы бойынша орташа SprintSpeed ең жоғары үш позицияны табыңыз. Жиынтық кестеден әр позицияның клубтық орташа мәндерінің салмақсыз орташа мәнін бөлек есептеңіз; реттеулерді салыстырып, ықтимал айырмаларды түсіндіріңіз. ST позициясы үшін орташа мәні ең жоғары үш клубты тауып, әр клубтағы ST санын көрсетіңіз.

### 9. Қорытындылар және экспорт

Орташа мәндердің жеке көрсетілімдік жиынтығына `margins=True` қосыңыз. Жалпы қорытындыны бастапқы бағанның тікелей орташа мәнімен және бос емес ұяшықтардың қарапайым орташа мәнімен салыстырыңыз: қандай теңдік міндетті, ал қайсысы талап етілмейтінін түсіндіріңіз. club_metrics, club_position_counts және club_position_speed экспорттаңыз. Жиынтық кестелерді экспорттар алдында Club индексін кәдімгі бағанға айналдырып, index=False арқылы сақтаңыз. Бастапқы жиынтық кестелердің санақ тексерулеріне қорытынды жолдарды қоспаңыз.

## Нәтижелер және тексерулер

lab04.py не lab04.ipynb, есеп және үш CSV тапсырыңыз. Есепте әр тапсырма бойынша формула не команда, бастапқы бақылаулар саны және мазмұнды түсіндірме болуы тиіс. Жиынтық кестелер үшін бар және жоқ үйлесімі көрінетін шағын үзінді көрсетіңіз. Барлық клубтың толық кең шығарылымын құжатқа енгізу қажет емес.

| Файл | Схема | Тексеру |
|---|---|---|
| lab04_club_metrics.csv | Club, n_players, wage_mean, wage_median, wage_min, wage_max | әр Club үшін бір жол |
| lab04_club_position_counts.csv | Club және позиция бағандары | жетіспейтін мән жоқ, санауыштар теріс емес |
| lab04_club_position_speed.csv | Club және позиция бағандары | бос үйлесімдер бос қалады |

Позиция бағандарының динамикалық жиыны келісілген сұрыптаудан кейін екі жиынтық кестеде сәйкес болуы тиіс. Қажетті позиция бағаны түгел болмаса, оны айқын хабарлаңыз немесе reindex арқылы кестені жарияланған позициялар тізіміне келтіріңіз. Көрсеткіш мағынасын ескермей барлық қорытынды файлға `.fillna(0)` қолдануға болмайды.

Күтілетін бақылау қасиеттері: санауыштар қосындысы бастапқы жазба сандарына тең; орташа мән мен медиана өз тобының ауқымында; min <= max; максимум бойынша таңдалған жол қажетті санаттар үйлесіміне шынымен жатады. Тәуелсіз сүзгілеу тексерулері әртүрлі есептің кемінде екі ұяшығын растауы тиіс.

## Қосымша тапсырма

Орташа SprintSpeed жиынтық кестесі үшін club_position_counts салмақтарын қолданып, әр позицияның жалпы көрсеткішін клубтық орташа мәндердің салмақталған орташа мәні ретінде қалпына келтіріңіз. Нәтижені тікелей groupby арқылы салыстырыңыз. SprintSpeed ішінде жетіспейтін мәндері бар файл үшін салмақ ретінде жалпы жол санын емес, бос емес SprintSpeed санын неге қолдану керектігін түсіндіріңіз.

## Қорғау сұрақтары

1. Орташа мән мен медиана теңдігі неліктен барлық мән тең дегенді білдірмейді?
2. pivot орнына pivot_table қашан қолдану керек?
3. Неліктен жоқ орташа SprintSpeed мәнін автоматты түрде нөлге алмастыруға болмайды?
4. GK жоқ клубтарды қосу GK орташа санауышына қалай әсер етеді?
5. Клубтық орташа мәндердің орташа мәні ойыншылар бойынша орташа мәннен неге ерекшеленеді?

## Бағалау

| Құрамдас бөлік | Балл |
|---|---|
| Club агрегаттары, mean/median теңдігі және көлемдерді ескеру | 20 |
| Argentina, Chelsea таңдамалары және максимумдарды тексеру | 20 |
| Санақ жиынтығы және қосындыларды бақылау | 20 |
| Орташа SprintSpeed, жетіспейтін мәндер және салмақ мағынасы | 20 |
| Экспорт және қайталанымдылық | 10 |
| Түсіндіру, есеп және қорғау | 10 |
| Барлығы | 100 |

## Дереккөздер

Негізі: берілген 4-зертханалық жұмыстың тапсырмалары және 4-дәрістің топтауды жалғастыру жөніндегі презентациясы. Жетіспейтін мәндерді, тең максимумдарды және орташалаудың әртүрлі деңгейлерін есепке алу ережелері нақтыланған.

- [pandas: groupby және agg](https://pandas.pydata.org/docs/user_guide/groupby.html).
- [pandas: жиынтық кестелер](https://pandas.pydata.org/docs/user_guide/reshaping.html).
- [pandas: pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html).


## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
